# Generate synthetic data for the agnews dataset
## Llama-2-7b-chat-hf

<https://huggingface.co/meta-llama/Llama-2-7b-hf>  
<https://huggingface.co/meta-llama/Llama-2-7b-chat-hf>

1. baseline
2. targeted + linguistic tags
3. unsupervised context
4. (unsupervised context + linguistic tags)

## Notes
### About chat template

the base *meta-llama/Llama-2-7b-hf* doesn't work quite right for the task of data generation. So I'm going to use the chat version.

After different tests I have seen that the chat version doesn't directly generate the text, but is going to ask for which label to you want it. So I'm going to randomly select a label each time and use the following format:

```python
<s>[INST] <<SYS>>
{{ system_prompt }}
<</SYS>>

{{ user_msg_1 }} [/INST] {{ model_answer_1 }} </s><s>[INST] {{ user_msg_2 }} [/INST]
```

with:
- model_answer_1 = "Of course! I'm happy to help. Please provide me with the category you would like me to focus on, and I will generate a high-quality short document for you."
- user_msg_2 = "label: " + random_label

The actual format with the correct special tokens is going to be generated by the `tokenizer.chat_template()`, we supply a list of messages of the kind:

```python
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_msg_1},
    {"role": "assistant", "content": model_answer_1},
    {"role": "user", "content": user_msg_2},
]
```

#### user_msg_2
- "The label is up to you.": the results tend to be very skewed towards a particular label.
- "label: " + random_label (the used one): works fine, but the cons is that we will have a uniform distribution. And that for the case of the unsupervised context we are "limiting" the options given by the unsupervised context. Instead of "picking" the best context example to look at, this will force a more strinct generation.

### About context examples

the context examples are inserted in the prompt where there is the placeholder `"ADD_CONTEXT_HERE"`, formatted as a bulled list.

There is a similar placeholder for inserting the random label: `"ADD_LABEL_HERE"`

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM, BitsAndBytesConfig

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")
from src._utils._generate_dataset import main_generate_dataset
from src._utils._helpers import get_generated_examples_df, clear_cuda_cache, get_context_examples

# get true labels
df_real = pd.read_csv("real_data/train/agnewstrainAll.csv").rename(
    columns={"2": "text", "3": "label"}
)
correct_labels = df_real["label"].unique().tolist()
labels_str = ", ".join(correct_labels)
labels_str_bullet = "\n".join([f"- {name}" for name in correct_labels])
model = None
HF_TOKEN = open("src/_utils/hf_token.txt","r").read() # your huggingface token

In [2]:
# chat template format:
# <s>[INST] <<SYS>>
# {{ system_prompt }}
# <</SYS>>
# 
# {{ user_msg_1 }} [/INST] {{ model_answer_1 }} </s><s>[INST] {{ user_msg_2 }} [/INST]

PROMPTS = {}
# same for all prompts
system = "You are an expert in journalism and NLP specializing in news classification."
assistant_response = "Of course! I'm happy to help. Please provide me with the category you would like me to focus on, and I will generate a high-quality short document for you."
user_response = "label: ADD_LABEL_HERE" # <- special token

####### BASELINE #######
prompt_baseline = f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

Choose one of the categories (labels), generate the corresponding text and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
}}
```
"""
PROMPTS["baseline"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_baseline},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]


####### TARGETED #######
prompt_targeted =  f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

For each example, also list the key phenomena it covers.

### **Follow these topics:**
- **Business**  
  - Markets  
  - Economy  
  - Companies  
  - Startups  
  - Regulations  

- **Sci/Tech**  
  - AI  
  - Space  
  - Cybersecurity  
  - Biotech  
  - Climate  

- **Sports**  
  - Events  
  - Records  
  - Highlights  
  - Scandals  
  - Olympics  

- **World**  
  - Politics  
  - Conflicts  
  - Disasters  
  - Human Rights  
  - Trade

### **Output Format (JSON)**
The labels must be one of the specified categories, which are: {labels_str}. \
Choose one of the categories (labels), generate the corresponding text, write the corresponding phenomena and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
    "phenomena": ["<phenomenon1>", "<phenomenon2>", ...]
}}
```
"""
PROMPTS["targeted + linguistic tags"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_targeted},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]


####### UNSUPERVISED CONTEXT #######
# 'ADD_CONTEXT_HERE' is a placeholder for the context that will be added at each iteration
prompt_unsupervised = f"""\
Your task is to generate an high-quality short documents (around 30 words), that talks about one of the following four News categories (labels):
{labels_str_bullet}

Here some examples of the documents you can use as a reference:
ADD_CONTEXT_HERE

Choose one of the categories (labels), generate the corresponding text and return it in the following JSON format:

```json
[
    {{
        "text": "<text of the document>", 
        "label": "<corresponding label>",
    }}
]
```
"""
PROMPTS["unsupervised context"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_unsupervised},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]

# Llama-2-7b-chat-hf

In [ ]:
#############################################
# LOAD MODEL
#############################################

if model:
    clear_cuda_cache(model)

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_name = "meta-llama/Llama-2-7b-chat-hf"
model = LlamaForCausalLM.from_pretrained(
            model_name, 
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            attn_implementation='flash_attention_2',
            quantization_config=quantization_config,
            low_cpu_mem_usage=True
        ).to("cuda")

tokenizer = LlamaTokenizer.from_pretrained(model_name, token=HF_TOKEN)

OUTPUT_DIR = "synthetic_data/datasets/Llama-2-7b-chat-hf/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
base_config = {
    "dataset": "agnews",
    "model": model,
    "tokenizer": tokenizer,
    # "generation_method": "baseline",
    #### We are going to give as input the list of messages
    # "prompt": {system:..., user: prompt, assistant: None},
    "apply_chat_template": True,
    "num_examples": 500,
    "max_new_tokens": 1024,
    "seed": 42,
    #"json_output_file": OUTPUT_DIR+"agnews_baseline_500.json",
    "log_file": OUTPUT_DIR+"generate_dataset_agnews_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label"],
    ### context
    # "context_examples": None,
    # "prompt_postfix": None,
    "verbose": False,
    ### At each iteration we ask randomly for a label
    "add_random_label": True, 
}

### 1. baseline

In [ ]:
name = "baseline"
config = base_config.copy()
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_baseline_500.json"
main_generate_dataset(config)

Generating Examples:   0%|          | 0/500 [00:00<?, ?ex/s]

Generating Examples:   4%|▍         | 21/500 [00:45<16:07,  2.02s/ex, examples=21/500, run=23]

❌ Failed to parse generation 23: Expecting ',' delimiter: line 3 column 5 (char 500)


Generating Examples:  28%|██▊       | 141/500 [05:01<12:00,  2.01s/ex, examples=141/500, run=144]

❌ Failed to parse generation 144: Expecting ',' delimiter: line 3 column 5 (char 542)


...

❌ Failed to parse generation 238: Invalid control character at: line 2 column 485 (char 486)


Generating Examples:  67%|██████▋   | 334/500 [12:04<06:31,  2.36s/ex, examples=334/500, run=341]

❌ Failed to parse generation 341: Expecting ',' delimiter: line 3 column 5 (char 445)


Generating Examples: 100%|██████████| 500/500 [17:56<00:00,  2.15s/ex, examples=500/500, run=510]

⏱️ Time taken: 1076.73 seconds.


### 2. targeted + linguistic tags

In [ ]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]
main_generate_dataset(config)

Generating Examples:   3%|▎         | 13/500 [00:45<23:40,  2.92s/ex, examples=13/500, run=15]

❌ Failed to parse generation 15: Expecting ',' delimiter: line 3 column 5 (char 563)


Generating Examples:  33%|███▎      | 166/500 [08:53<16:03,  2.89s/ex, examples=166/500, run=169]

❌ Failed to parse generation 169: Expecting ',' delimiter: line 3 column 5 (char 491)


Generating Examples:  38%|███▊      | 191/500 [10:14<15:47,  3.07s/ex, examples=191/500, run=195]

...

❌ Failed to parse generation 341: Expecting ',' delimiter: line 3 column 5 (char 505)


Generating Examples:  95%|█████████▍| 474/500 [24:57<01:11,  2.75s/ex, examples=474/500, run=481]

❌ Failed to parse generation 481: Expecting ',' delimiter: line 3 column 5 (char 400)


Generating Examples: 100%|██████████| 500/500 [26:20<00:00,  3.16s/ex, examples=500/500, run=507]

⏱️ Time taken: 1580.09 seconds.


In [ ]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]

In [ ]:
### Generate other 500 examples

config['seed'] = (config['seed'] + 1)*8
config['json_output_file'] = OUTPUT_DIR+"agnews_targeted+tags_500_2.json"
main_generate_dataset(config)

Generating Examples: 100%|██████████| 5/5 [00:33<00:00,  6.65s/ex, examples=5/5, run=6]

⏱️ Time taken: 33.24 seconds.


### 3. unsupervised context

In each prompt we attach n (5) examples sampled randomly from the train set. The samples are used without the labels, so they works as unsupervised context for the model, when we will generate the new synthetic sample. 

In [5]:
# we take more than 500 because it can happen that some prompt
# generate the example in the wrong format, so is not read correctly (and discarded)
num_prompts = 1000
num_examples_per_prompt = 5
np.random.seed(42)
context_examples = get_context_examples(df_real, num_examples_per_prompt, num_prompts)
print(f"Number prompts: {len(context_examples)}")
print(f"Number of examples per prompt: {len(context_examples[0])}")
print(context_examples[0])

Number prompts: 1000
Number of examples per prompt: 5
['With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.', 'Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify ', 'King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.', 'LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but activity is modest as firms still fear calling a premature end to this year #39;s stunning price rise, traders said on Friday. ', 'STOCKHOLM (AFP) - Andre Agassi was set to intensify his chase for 

In [ ]:
config = base_config.copy()
name = "unsupervised context"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_unsupervisedContext_500.json"
config["context_examples"] = context_examples
main_generate_dataset(config)

In [ ]:
generated_df, _ = get_generated_examples_df(OUTPUT_DIR+"agnews_unsupervisedContext_500.json")

for i in range(5):
    print("TEXT: "+generated_df.iloc[i]['text'])
    print("LABEL: "+generated_df.iloc[i]['label'])
    print("CONTEXT EXAMPLES:")
    for j in range(len(generated_df.iloc[i]['context_examples'])):
        print("- "+generated_df.iloc[i]['context_examples'][j])

    print("\n"+"=="*50)

TEXT: LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but activity is modest as firms still fear calling a premature end to this year's stunning price rise, traders said on Friday. #oilprices #energy
LABEL: Business
CONTEXT EXAMPLES:
- With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.
- Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify 
- King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.
- LONDON (Reuters) - Oil producers are emerging to lock in record high pri

In [ ]:
# Unsupervised context + tags
# ...